<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261013/%E3%82%AF%E3%83%A9%E3%82%B9%E3%82%BF%E3%83%AA%E3%83%B3%E3%82%B0%E3%81%AE%E6%BC%94%E7%BF%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#演習

##実行環境の準備



In [ ]:
%%capture
!pip install japanize-matplotlib
!pip -q install kagglehub

In [ ]:
# 必要なライブラリの準備
import os
import glob
import kagglehub

import sys
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#前処理として標準化を行う。そのためのモジュールをインポートする
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 日本語表示用ライブラリを利用
import japanize_matplotlib

# 表示設定
pd.options.display.float_format = "{:.2f}".format
sns.set_theme(style='whitegrid', font='IPAexGothic')

plt.rcParams["figure.figsize"] = (8, 5)
#マイナス表示に Unicode U+2212 を使わない（ASCII 半角ハイフン U+002D を使用させる）
plt.rcParams["axes.unicode_minus"] = False

# 再現性を確保するための乱数シード
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

#使用するデータ

本演習では、Kaggleで公開されている **Wholesale customers Data Set** を使用する。

- Kaggle: `binovi/wholesale-customers-data-set`
- Kaggle URL: https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set

このデータは、ある卸売業者の顧客について、**6つの商品カテゴリへの年間支出額**と、  
顧客の販売チャネル・地域を記録したものである。

データは **440顧客、8変数**から構成される。

#分析のテーマ

6つの商品カテゴリへの年間支出額を顧客別に集計したデータを使い
顧客をセグメントに分割し、営業・マーケティング施策を提案したい。
ビジネス的な活用が可能なクラスタリングを行え。


###分析に際する注意点

K-means 法によるクラスタリングでは、ユークリッド距離を使うため、極端な値はクラスタ中心を強く引っ張る可能性がある。データには大口顧客が含まれており、適切な処理が求めらられる。

#データの取得

以下のセルではKaggleからデータを取得する。
公開データであるため通常はそのまま取得できるが、
エラーとなる場合は、Kaggleから `Wholesale customers data.csv` をダウンロードし、手動で読み込む。

In [ ]:
DATASET_HANDLE = "binovi/wholesale-customers-data-set"

dataset_dir = kagglehub.dataset_download(DATASET_HANDLE)

csv_files = glob.glob(
    os.path.join(dataset_dir, "**", "*.csv"),
    recursive=True
)

preferred = [
    p for p in csv_files
    if "wholesale" in os.path.basename(p).lower()
]

csv_path = preferred[0] if preferred else csv_files[0]
df_raw = pd.read_csv(csv_path)

print("Kaggleから読み込みました。")
print("ファイル:", csv_path)

##データ項目の説明

| 変数 | 内容 | 本演習での扱い |
|---|---|---|
| `Fresh` | 生鮮食品への年間支出額 | クラスタリングに使用 |
| `Milk` | 乳製品への年間支出額 | クラスタリングに使用 |
| `Grocery` | 食料雑貨への年間支出額 | クラスタリングに使用 |
| `Frozen` | 冷凍食品への年間支出額 | クラスタリングに使用 |
| `Detergents_Paper` | 洗剤・紙製品への年間支出額 | クラスタリングに使用 |
| `Delicassen` | デリカテッセンへの年間支出額 | クラスタリングに使用 |
| `Channel` | 顧客チャネル | **クラスタリングには使わず、事後比較に使用** |
| `Region` | 顧客地域 | **クラスタリングには使わず、事後比較に使用** |

### Channel のコード

- `1`: Horeca（Hotel / Restaurant / Café）
- `2`: Retail

### Region のコード

- `1`: Lisbon
- `2`: Oporto
- `3`: Other


In [ ]:
# 元データを保持し、分析用DataFrameを作成する
df = df_raw.copy()
df.columns = df.columns.str.strip()

# 解釈用のラベル列を追加する
channel_map = {
    1: "Horeca", # Hotel/Restaurant/Cafe
    2: "Retail"  # Retail channel
}

region_map = {
    1: "Lisbon",
    2: "Oporto",
    3: "Other"
}

df["Channel_Label"] = df["Channel"].map(channel_map)
df["Region_Label"] = df["Region"].map(region_map)

display(df.head())